In [1]:
import torch
import torch.nn as nn

class SimpleViT(nn.Module):

    def __init__(
        self,
        image_size=32,
        patch_size=8,
        dim=64
    ):

        super().__init__()

        self.patch = nn.Conv2d(
            3,
            dim,
            patch_size,
            patch_size
        )

        num_patches = (
            image_size // patch_size
        ) ** 2

        self.cls = nn.Parameter(
            torch.randn(1, 1, dim)
        )

        self.pos = nn.Parameter(
            torch.randn(
                1,
                num_patches + 1,
                dim
            )
        )

        self.attention = nn.MultiheadAttention(
            dim,
            4,
            batch_first=True
        )

    def forward(self, x):

        x = self.patch(x)

        x = x.flatten(2).transpose(1, 2)

        cls = self.cls.expand(
            x.size(0), -1, -1
        )

        x = torch.cat([cls, x], dim=1)

        x = x + self.pos

        x, attention = self.attention(
            x, x, x
        )

        return x, attention


model = SimpleViT()

image = torch.randn(
    2, 3, 32, 32
)

output, attention = model(image)

print("Output shape:", output.shape)
print("Attention shape:", attention.shape)

Output shape: torch.Size([2, 17, 64])
Attention shape: torch.Size([2, 17, 17])
